# 20 · Decorators

**Topics covered:** functions as arguments · nested functions · decorator basics · `@` syntax · `*args/**kwargs` decorators · `functools.wraps` · decorator as **scheduler** · decorator as **validator** · decorators with arguments

## 1. What is a decorator?
A decorator **extends the functionality of an existing function without changing its original code**.
Typical uses: logging, timing, scheduling, validation, authentication.

## 2. Step 1 – pass a function as an argument (functions are objects)

In [1]:
def test():
    print("test() function is called")

def demo(func):
    print("demo() function is called")
    func()

demo(test)

demo() function is called
test() function is called


## 3. Step 2 – the ATM example (without decorator)

In [2]:
def withdraw(amount):
    print(f"{amount} withdrawn successfully")

withdraw(25000)

# adding extra work (pin / receipt) by hand every time:
def ATM_plain(func, amount):
    print("pin")
    func(amount)
    print("receipt")

ATM_plain(withdraw, 25000)

25000 withdrawn successfully
pin
25000 withdrawn successfully
receipt


## 4. Step 3 – decorator = outer function + inner (wrapper) function
1. outer function **receives** a function
2. inner **wrapper** adds extra behaviour and calls the original
3. outer function **returns the wrapper** (not calling it)

In [3]:
def ATM(func):                   # decorator
    def insidefunc(amount):      # wrapper
        print("pin")
        func(amount)
        print("receipt")
    return insidefunc            # return wrapper (no brackets)

f1 = ATM(withdraw)               # manual decoration
f1(25000)

pin
25000 withdrawn successfully
receipt


In [4]:
def smallfunc():
    print("middle")

def decofunc(f):
    def insidefunc():            # wrapper func
        print("start")
        f()
        print("end")
    return insidefunc            # wrapper returned

d = decofunc(smallfunc)
d()

start
middle
end


## 5. The `@` syntax (annotation)
`@decorator` above a function is shortcut for `func = decorator(func)`.

In [5]:
def decofunc(f):
    def insidefunc():
        print("start")
        f()
        print("end")
    return insidefunc

@decofunc
def smallfunc():
    print("middle")

smallfunc()

start
middle
end


In [6]:
def ATM(func):
    def insidefunc():
        print("pin")
        func()
        print("receipt")
    return insidefunc

@ATM
def withdraw():
    print("withdrawn successfully")

withdraw()

pin
withdrawn successfully
receipt


### ⚠️ Common mistake – decorator without a wrapper
In your notebook the last `ATM` version ran `func()` **immediately when the function was defined** (at decoration time) and not each time you call it:

In [7]:
def ATM_wrong(func):
    print("pin")
    func()
    print("receipt")
    return func              # runs once, at definition time!

@ATM_wrong
def withdraw():
    print("Money withdrawn")

print("---- now calling withdraw() ----")
withdraw()                   # only the original runs; no pin / receipt this time

pin
Money withdrawn
receipt
---- now calling withdraw() ----
Money withdrawn


## 6. Decorating functions with any arguments – `*args, **kwargs`

In [8]:
def intro(func):
    def wrapper(*args, **kwargs):
        print("hi")
        func(*args, **kwargs)
        print("Welcome to Agentic AI")
    return wrapper

@intro
def sample(*args, **kwargs):
    for data in args:
        print(data)
    for name, value in kwargs.items():
        print(name, value)

sample(20, 30, name="rajesh", education="Btech", Position="AI Eng.")

hi
20
30
name rajesh
education Btech
Position AI Eng.
Welcome to Agentic AI


## 7. Returning values & keeping metadata – `functools.wraps`

In [9]:
from functools import wraps

def log(func):
    @wraps(func)                      # keeps original __name__ and docstring
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__} with {args} {kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result                 # don't forget to return!
    return wrapper

@log
def add(a, b):
    """adds two numbers"""
    return a + b

add(3, 4)
print(add.__name__, "|", add.__doc__)

calling add with (3, 4) {}
add returned 7
add | adds two numbers


## 8. Decorator as a **scheduler**
Allow the task only between 10:00 and 18:00.

In [10]:
import datetime

def Scheduletask(func):
    def wrapper():
        hour1 = datetime.datetime.now().hour
        if hour1 >= 10 and hour1 <= 18:
            func()
            print("current hour:", hour1)
        else:
            print("Task is not allowed at this time!!!")
    return wrapper

@Scheduletask
def Test():
    print("Task is running")

Test()         # result depends on the time you run it

Task is not allowed at this time!!!


In [11]:
# testable version: pass the hour so we can see both outcomes
def schedule(start, end):
    def decorator(func):
        def wrapper(hour):
            if start <= hour <= end:
                return func()
            print(f"Task not allowed at {hour}:00 (allowed {start}-{end})")
        return wrapper
    return decorator

@schedule(10, 18)
def job():
    print("Task is running")

job(9)
job(14)

Task not allowed at 9:00 (allowed 10-18)
Task is running


## 9. Decorator as a **validator**

In [12]:
# Exercise: validator() to validate age; print "Eligible to vote" if age is 18 or more
def validator(func):
    def wrapper(age):
        if isinstance(age, int) and age >= 18:
            print("Eligible to vote")
            func(age)
        else:
            print("Not eligible to vote")
    return wrapper

@validator
def vote(age):
    print(f"Vote registered for age {age}")

vote(25)
vote(16)

Eligible to vote
Vote registered for age 25
Not eligible to vote


In [13]:
def Valid_Student(Function):
    def InsideFunction(Name, RollNo):
        if isinstance(Name, str) and len(Name) <= 8:
            Function(Name, RollNo)
        else:
            print("Invalid Name")
    return InsideFunction

@Valid_Student
def Get_Details(Name, RollNo):
    print("Student Name is", Name)
    print("Student Roll is", RollNo)

Name = str(input("Enter Name: "))
RollNo = int(input("Enter Roll: "))
Get_Details(Name, RollNo)

Enter Name: shubham
Enter Roll: 20
Student Name is shubham
Student Roll is 20


In [14]:
# same decorator, name too long
Get_Details("shubhamjadhav", 21)

Invalid Name


## 10. Decorators with arguments (decorator factory)

In [15]:
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(3)
def hello(name):
    print("Hello", name)

hello("Shubham")

Hello Shubham
Hello Shubham
Hello Shubham


## 11. Timing decorator (real-world use)

In [16]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - t0:.5f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))

slow_sum took 0.01232s
499999500000


## 12. Stacking decorators
Applied bottom-up: `@A @B def f` ⇒ `f = A(B(f))`.

In [17]:
def bold(f):
    return lambda: "<b>" + f() + "</b>"
def italic(f):
    return lambda: "<i>" + f() + "</i>"

@bold
@italic
def text():
    return "hello"

print(text())

<b><i>hello</i></b>
